# Hotdog / Not Hotdog: the CNN

Fine-tunes an ImageNet-pretrained **ConvNeXt-Tiny** to tell hot dogs from everything else and exports it for the
CNN server. Run it here cell by cell, or headless with `./run.sh train` (~20 min on an Apple Silicon Mac).

**Data:** [Food-101](https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/), a free ~5 GB download with no
account needed. Training uses its 750 training hot dogs plus ~2 200 other dishes, two thirds of them look-alikes (fries,
prime rib, tacos …). The test set is 250 hot dogs + 250 other dishes from Food-101's separate test split.

**Steps:** 1 setup · 2 configuration · 3 data · 4 augmentation and loaders · 5 model · 6 training · 7 calibration ·
8 evaluation · 9 export.

`SMOKE=1` runs 2 epochs on synthetic images without any download, to check the pipeline.

## 1. Setup

Imports, plus the two helper modules next to this notebook: `hotdog_model.py` (architectures, shared with the server)
and `hotdog_data.py` (dataset loaders).

In [ ]:
import json
import math
import os
import platform
import random
import sys
import time
import warnings
from collections import Counter
from contextlib import nullcontext
from datetime import UTC, datetime
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm

sys.path.insert(0, str(Path.cwd()))  # the notebook runs from cnn/, next to the helper modules
from hotdog_data import (
    HotdogDataset,
    drop_overlaps,
    load_folder,
    load_food101,
    load_food101_splits,
    load_hf,
    load_kaggle,
    make_synthetic,
    open_image,
)
from hotdog_model import (
    ARCHS,
    CLASSES,
    IMAGENET_MEAN,
    IMAGENET_STD,
    IMG_SIZE,
    RESIZE_SIZE,
    build_model,
    cam_layer,
    count_params,
    describe,
    param_groups,
    pick_device,
    set_backbone_trainable,
    sync_device,
)

print(f"python {platform.python_version()} | torch {torch.__version__}")

## 2. Configuration

All settings live in this cell. Those in the table can also be set as environment variables of the same name
(e.g. `ARCH=efficientnet_b0 ./run.sh train`).

| Setting | Options |
|---|---|
| `DATA_SOURCE` | `food101` (default, no account) · `kaggle` (SeeFood, needs Kaggle credentials) · `hf` (a Hugging Face dataset) · `folder` (`data/{train,test}/{hot_dog,not_hot_dog}`) |
| `ADD_FOOD101` | for `kaggle` / `hf` / `folder` only: `1` (default) adds Food-101 to training · `0`: source only |
| `ARCH` | `convnext_tiny` (default) · `efficientnet_b0` · `mobilenet_v3_large` (faster, lighter) |
| `PRETRAINED` | `1` (default): start from ImageNet weights · `0`: random init |
| `DEVICE` | auto (CUDA → Apple MPS → CPU) or e.g. `cpu` |
| `SMOKE` | `1`: 2 quick epochs on synthetic images, no downloads |
| `DATA_DIR`, `CNN_ARTIFACTS` | where data is read/downloaded (`data`) and where the model is written (`artifacts`) |

In [ ]:
def env(name: str, default: str) -> str:
    return os.environ.get(name) or default


SEED = 42
SMOKE = env("SMOKE", "0") == "1"

# data
DATA_SOURCE = env("DATA_SOURCE", "food101")  # "food101" | "kaggle" | "hf" | "folder"
DATA_DIR = Path(env("DATA_DIR", "data"))  # folder source + Food-101 download location
KAGGLE_ID = "dansbecker/hot-dog-not-hot-dog"
HF_DATASET, HF_IMAGE_COL, HF_LABEL_COL, HF_TEST_SPLIT = "truepositive/hotdog_nothotdog", "image", "label", "test"
ADD_FOOD101 = env("ADD_FOOD101", "1") == "1" and DATA_SOURCE != "food101" and not SMOKE
FOOD101_HOTDOGS = 1000  # all of Food-101's hot_dog class (only with ADD_FOOD101)
FOOD101_HARD_PER_CLASS = 60  # x ~25 look-alike classes (~1 500 negatives)
FOOD101_OTHER_PER_CLASS = 10  # x ~75 other classes      (~  750 negatives)
VAL_FRACTION = 0.15

# model
ARCH = env("ARCH", "convnext_tiny")
assert ARCH in ARCHS, f"ARCH must be one of {ARCHS}"
PRETRAINED = env("PRETRAINED", "1") == "1" and not SMOKE  # smoke mode never downloads

# training
EPOCHS, FREEZE_EPOCHS, WARMUP_EPOCHS, PATIENCE = 12, 2, 1, 4  # phase 1 (head only) is FREEZE_EPOCHS long
LR, FINETUNE_LR, BACKBONE_LR_MULT = 1e-3, 5e-4, 0.2  # phase 2: backbone fine-tunes at 5e-4 * 0.2 = 1e-4
BATCH_SIZE = 32
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.05  # light; calibration is done properly in section 7
GRAD_CLIP = 2.0
USE_AMP = True  # fp16 autocast on CUDA / MPS when supported
if SMOKE:
    EPOCHS, FREEZE_EPOCHS, WARMUP_EPOCHS, PATIENCE, BATCH_SIZE = 2, 1, 0, 2, 8

# hardware and output
device = pick_device(env("DEVICE", "") or None)
NUM_WORKERS = 0 if SMOKE else min(8, os.cpu_count() or 1)
ARTIFACTS_DIR = Path(env("CNN_ARTIFACTS", "artifacts"))

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)  # seeds CUDA / MPS too
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True  # fixed input size: let cuDNN pick the fastest kernels

print(
    f"SMOKE={SMOKE} | data={'synthetic' if SMOKE else DATA_SOURCE}{' + Food-101' if ADD_FOOD101 else ''} | "
    f"ARCH={ARCH} ({describe(ARCH)}) | pretrained={PRETRAINED} | epochs={EPOCHS} | "
    f"device={device} | workers={NUM_WORKERS}"
)

## 3. Data

Each source yields training images (with a *source* tag such as `food101:tacos`, used for reporting) and test images.
The test split is only used in section 8. Food-101's train and test splits never overlap. With another source plus
`ADD_FOOD101`, Food-101 images matching any of that source's images (same file id or near-identical perceptual hash)
are dropped first.

In [ ]:
print("==> Data")
if SMOKE:
    (train_x, train_y), (test_x, test_y) = make_synthetic(48, 1), make_synthetic(16, 2)
    train_src, data_desc = ["synthetic"] * len(train_x), "synthetic (SMOKE)"
elif DATA_SOURCE == "food101":
    (train_x, train_y, train_cls), (test_x, test_y) = load_food101_splits(
        DATA_DIR, FOOD101_HARD_PER_CLASS, FOOD101_OTHER_PER_CLASS, seed=SEED
    )
    train_src, data_desc = [f"food101:{c}" for c in train_cls], "food101"
else:
    loaders = {
        "kaggle": (partial(load_kaggle, KAGGLE_ID), f"kaggle:{KAGGLE_ID}"),
        "hf": (partial(load_hf, HF_DATASET, HF_IMAGE_COL, HF_LABEL_COL, HF_TEST_SPLIT), f"hf:{HF_DATASET}"),
        "folder": (partial(load_folder, DATA_DIR), f"folder:{DATA_DIR}"),
    }
    if DATA_SOURCE not in loaders:
        raise ValueError(f"unknown DATA_SOURCE {DATA_SOURCE!r}; choose food101, {', '.join(loaders)}")
    load, data_desc = loaders[DATA_SOURCE]
    (train_x, train_y), (test_x, test_y) = load()
    train_src = [DATA_SOURCE] * len(train_x)

if len(test_x) == 0:  # source without a test split: carve one out before anything else touches the data
    train_x, test_x, train_y, test_y, train_src, _ = train_test_split(
        train_x, train_y, train_src, test_size=0.2, stratify=train_y, random_state=SEED
    )

if ADD_FOOD101:
    t0 = time.time()
    f_x, f_y, f_cls = load_food101(
        DATA_DIR, FOOD101_HOTDOGS, FOOD101_HARD_PER_CLASS, FOOD101_OTHER_PER_CLASS, seed=SEED
    )
    f_x, f_y, f_cls, dropped = drop_overlaps(f_x, f_y, f_cls, protected=[*test_x, *train_x])
    print(
        f"Food-101: kept {len(f_x)} images, removed overlaps with {DATA_SOURCE}: {dropped or 'none'} ({time.time() - t0:.0f}s)"
    )
    train_x, train_y = [*train_x, *f_x], [*train_y, *f_y]
    train_src = [*train_src, *(f"food101:{c}" for c in f_cls)]
    data_desc += " + food101"

### Validation split

15 % of the training pool becomes the validation set, used for model selection and calibration. The split is
stratified by label *and* source, so every source is represented in both parts.

In [ ]:
strata = [f"{y}|{s.split(':')[0]}" for y, s in zip(train_y, train_src, strict=True)]
idx_tr, idx_val = train_test_split(np.arange(len(train_x)), test_size=VAL_FRACTION, stratify=strata, random_state=SEED)


def take(seq, idx):
    return [seq[i] for i in idx]


val_x, val_y, val_src = take(train_x, idx_val), take(train_y, idx_val), take(train_src, idx_val)
train_x, train_y, train_src = take(train_x, idx_tr), take(train_y, idx_tr), take(train_src, idx_tr)

for name, ys in [("train", train_y), ("val", val_y), ("test", test_y)]:
    n_hot, n_not = np.bincount(ys, minlength=2)
    print(f"{name:5s}: {len(ys):5d} images  hotdog={n_hot:5d}  not_hotdog={n_not:5d}")

### A look at the data

In [ ]:
def show_grid(images, titles, suptitle, ncols=6, size=2.2):
    nrows = max(1, math.ceil(len(images) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * size, nrows * size + 0.4), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, im, title in zip(axes.flat, images, titles, strict=False):  # spare axes stay blank
        ax.imshow(im)
        ax.set_title(title, fontsize=8)
    fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()


def thumbnail(src, size=160):
    return open_image(src).resize((size, size))


idx = random.Random(0).sample(range(len(train_x)), k=min(12, len(train_x)))
show_grid(
    [thumbnail(train_x[i]) for i in idx],
    [f"{CLASSES[train_y[i]]}\n{train_src[i]}" for i in idx],
    "Training samples (label / source)",
)

## 4. Augmentation and loaders

Training images get random crops, flips, TrivialAugment and random erasing, so the model sees a new variant of each
image every epoch. Evaluation uses `Resize(256) + CenterCrop(224)`, exactly like the server.

In [ ]:
train_tf = transforms.Compose(
    [
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.35, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.TrivialAugmentWide(),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.2)),
    ]
)
view_tf = transforms.Compose([transforms.Resize(RESIZE_SIZE), transforms.CenterCrop(IMG_SIZE)])  # also for plots
eval_tf = transforms.Compose([view_tf, transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

A `WeightedRandomSampler` draws every batch roughly 50/50 hot dog / not hot dog, although the pool has about three
times more negatives. Worker processes decode images in parallel; pinned memory speeds up copies to a CUDA GPU.

In [ ]:
class_counts = np.bincount(train_y, minlength=2)
sampler = WeightedRandomSampler(
    weights=[1.0 / class_counts[y] for y in train_y], num_samples=len(train_y), replacement=True
)
loader_kw = {"num_workers": NUM_WORKERS, "pin_memory": device.type == "cuda"}
train_dl = DataLoader(
    HotdogDataset(train_x, train_y, train_tf),
    batch_size=BATCH_SIZE,
    sampler=sampler,
    drop_last=len(train_x) > BATCH_SIZE,
    persistent_workers=NUM_WORKERS > 0,  # reused every epoch
    **loader_kw,
)
val_dl = DataLoader(
    HotdogDataset(val_x, val_y, eval_tf), batch_size=BATCH_SIZE * 2, persistent_workers=NUM_WORKERS > 0, **loader_kw
)
test_ds = HotdogDataset(test_x, test_y, eval_tf)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, **loader_kw)  # used once


def denorm(t):
    # normalized (3, H, W) tensor -> displayable (H, W, 3) array
    mean, std = torch.tensor(IMAGENET_MEAN).view(3, 1, 1), torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()


xb, yb = next(iter(train_dl))
show_grid(
    [denorm(t) for t in xb[:12]],
    [CLASSES[y] for y in yb[:12]],
    f"Augmented training batch ({(yb == 0).float().mean():.0%} hotdogs)",
)

## 5. Model

An ImageNet-pretrained CNN whose 1000-class output layer is replaced by a new 2-way head. On CUDA the network runs in
`channels_last` memory layout, which is faster for convolutions on tensor cores.

In [ ]:
MEMORY_FORMAT = torch.channels_last if device.type == "cuda" else torch.contiguous_format
model = build_model(ARCH, num_classes=len(CLASSES), pretrained=PRETRAINED).to(device, memory_format=MEMORY_FORMAT)
print(f"{ARCH}: {describe(ARCH)} | {count_params(model)[0]:,} parameters | pretrained: {PRETRAINED}")

## 6. Training

Building blocks first:

* **Mixed precision:** forward passes run in fp16 where the device supports it (checked with one probe batch). On CUDA a
  gradient scaler keeps small fp16 gradients from underflowing.
* **Optimizer:** AdamW with a short linear warm-up, then a cosine decay, stepped every batch.
* **Prediction:** `predict_logits` runs a loader through the model in `torch.inference_mode()` (no autograd
  bookkeeping) and is reused for validation, calibration and testing.

In [ ]:
def autocast_available() -> bool:
    if not USE_AMP or device.type not in ("cuda", "mps"):
        return False
    try:
        model.eval()  # probe in eval mode so BatchNorm running stats are untouched
        with torch.inference_mode(), torch.autocast(device.type, dtype=torch.float16):
            model(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=device))
        return True
    except Exception as e:
        print(f"autocast unavailable on {device.type} ({e}); using fp32")
        return False


AMP = autocast_available()
autocast = partial(torch.autocast, device.type, dtype=torch.float16) if AMP else nullcontext
scaler = torch.amp.GradScaler("cuda", enabled=AMP and device.type == "cuda")  # a no-op when disabled
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
print("mixed precision:", AMP)


def to_device(x, y):
    return x.to(device, non_blocking=True, memory_format=MEMORY_FORMAT), y.to(device, non_blocking=True)


def make_optimizer(lr, epochs, warmup_epochs, backbone_mult):
    opt = torch.optim.AdamW(param_groups(model, lr, backbone_mult), weight_decay=WEIGHT_DECAY)
    total, warm = max(1, epochs * len(train_dl)), warmup_epochs * len(train_dl)

    def lr_factor(step):
        if step < warm:
            return (step + 1) / warm
        return 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))

    return opt, torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)


def predict_logits(dl, m=None):
    # fp32 CPU logits and labels for a whole loader, in loader order
    m = m or model
    m.eval()
    logits, labels = [], []
    with torch.inference_mode():
        for x, y in dl:
            with autocast():
                logits.append(m(to_device(x, y)[0]).float().cpu())
            labels.append(y)
    return torch.cat(logits).clone(), torch.cat(labels)  # clone: a normal tensor, usable with autograd later

One epoch = one pass over the balanced sampler, then a validation pass. The epoch with the best validation accuracy
(ties: lower loss) is kept along with its validation logits, which section 7 reuses. A phase stops early after
`PATIENCE` epochs without improvement.

In [ ]:
history = {k: [] for k in ("train_loss", "train_acc", "val_loss", "val_acc", "lr")}
best = {"key": (-1.0, -math.inf), "val_acc": -1.0, "epoch": -1, "state": None, "val_logits": None, "val_ys": None}


def train_one_epoch(opt, sched, desc):
    model.train()
    loss_sum = torch.zeros((), device=device)  # accumulated on the device: no GPU sync per batch
    correct = torch.zeros((), device=device, dtype=torch.long)
    n = 0
    for x, y in tqdm(train_dl, desc=desc, leave=False):
        x, y = to_device(x, y)
        opt.zero_grad(set_to_none=True)
        with autocast():
            logits = model(x)
        loss = criterion(logits.float(), y)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(opt)
        scaler.update()
        sched.step()
        loss_sum += loss.detach() * len(y)
        correct += (logits.argmax(1) == y).sum()
        n += len(y)
    return loss_sum.item() / n, correct.item() / n


def run_phase(name, epochs, lr, warmup_epochs, backbone_mult):
    opt, sched = make_optimizer(lr, epochs, warmup_epochs, backbone_mult)
    stale = 0
    for ep in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(opt, sched, f"[{name}] epoch {ep}/{epochs}")
        val_logits, val_ys = predict_logits(val_dl)
        val_loss = F.cross_entropy(val_logits, val_ys).item()  # plain NLL, no label smoothing
        val_acc = (val_logits.argmax(1) == val_ys).float().mean().item()
        for k, v in zip(history, (train_loss, train_acc, val_loss, val_acc, sched.get_last_lr()[0]), strict=True):
            history[k].append(v)

        key = (round(val_acc, 4), -val_loss)
        improved = key > best["key"]
        if improved:
            state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best.update(key=key, val_acc=val_acc, epoch=len(history["val_acc"]), state=state)
            best.update(val_logits=val_logits, val_ys=val_ys)
            stale = 0
        else:
            stale += 1
        print(
            f"[{name}] epoch {ep:3d}  train loss {train_loss:.4f} acc {train_acc:.3f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.3f}{'  *' if improved else ''}"
        )
        if stale >= PATIENCE:
            print(f"early stopping: no val improvement for {PATIENCE} epochs")
            break

**Phase 1** trains only the new head while the pretrained backbone is frozen, so the random head cannot wreck the
pretrained features. **Phase 2** unfreezes everything; the backbone gets a smaller learning rate
(`BACKBONE_LR_MULT`) than the head.

In [ ]:
print("==> Training")
t_start = time.time()

set_backbone_trainable(model, False)
print(f"phase 1: head only ({count_params(model)[1]:,} trainable params)")
run_phase("head", FREEZE_EPOCHS, LR, warmup_epochs=0, backbone_mult=0.0)

set_backbone_trainable(model, True)
print(
    f"phase 2: full fine-tune ({count_params(model)[1]:,} trainable params, backbone LR {FINETUNE_LR * BACKBONE_LR_MULT:.0e})"
)
run_phase("finetune", EPOCHS - FREEZE_EPOCHS, FINETUNE_LR, warmup_epochs=WARMUP_EPOCHS, backbone_mult=BACKBONE_LR_MULT)

train_minutes = (time.time() - t_start) / 60
model.load_state_dict(best["state"])
print(f"\nbest val acc {best['val_acc']:.3f} at epoch {best['epoch']} | training took {train_minutes:.1f} min")

In [ ]:
epochs_axis = range(1, len(history["val_acc"]) + 1)
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(12, 4))
for split in ("train", "val"):
    ax_loss.plot(epochs_axis, history[f"{split}_loss"], label=split)
    ax_acc.plot(epochs_axis, history[f"{split}_acc"], label=split)
ax_loss.set(title="Loss (train includes label smoothing)", xlabel="epoch")
ax_acc.axvline(best["epoch"], ls="--", c="gray", lw=1, label="best")
ax_acc.axvspan(0.5, FREEZE_EPOCHS + 0.5, color="gray", alpha=0.1, label="head only")
ax_acc.set(title="Accuracy", xlabel="epoch", ylim=(0, 1.02))
ax_loss.legend()
ax_acc.legend()
plt.tight_layout()
plt.show()

## 7. Calibration

Temperature scaling fits one number `T` on the validation set so that `softmax(logits / T)` is honest: "90 % sure"
means right ~90 % of the time. Accuracy is unchanged. ECE (expected calibration error) groups predictions by confidence
and averages the gap between confidence and accuracy per group. The server applies `T`.

In [ ]:
def calibration_bins(probs, ys, n_bins=10):
    # per non-empty confidence bin (0.0-0.1], ..., (0.9-1.0]: (mean confidence, accuracy, count)
    conf, correct = probs.max(1), probs.argmax(1) == ys
    bin_idx = np.clip(np.ceil(conf * n_bins).astype(int) - 1, 0, n_bins - 1)
    return [(conf[bin_idx == b].mean(), correct[bin_idx == b].mean(), (bin_idx == b).sum()) for b in np.unique(bin_idx)]


def ece(probs, ys):
    return float(sum(n * abs(acc - conf) for conf, acc, n in calibration_bins(probs, ys)) / len(ys))


val_logits, val_ys = best["val_logits"], best["val_ys"]  # from the best epoch: no extra pass needed
log_t = torch.zeros(1, requires_grad=True)  # optimize log(T) so T stays positive
opt_t = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)


def nll_closure():
    opt_t.zero_grad()
    loss = F.cross_entropy(val_logits / log_t.exp(), val_ys)
    loss.backward()
    return loss


opt_t.step(nll_closure)
TEMPERATURE = float(log_t.exp().clamp(0.05, 20).item())

ece_before = ece(val_logits.softmax(1).numpy(), val_ys.numpy())
ece_after = ece((val_logits / TEMPERATURE).softmax(1).numpy(), val_ys.numpy())
print(f"temperature T = {TEMPERATURE:.3f}  |  val ECE {ece_before:.3f} -> {ece_after:.3f}")

## 8. Evaluation

The held-out test images, scored like the server does (calibrated probabilities). The test set is run through the
model once; all metrics and plots below reuse those predictions. Precision, recall and F1 treat **hotdog** as positive.

In [ ]:
print("==> Evaluation")
test_logits, test_ys = predict_logits(test_dl)
y_prob = (test_logits / TEMPERATURE).softmax(1).numpy()
y_true, y_pred = test_ys.numpy(), y_prob.argmax(1)

test_acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, pos_label=0, average="binary", zero_division=0)
auc = roc_auc_score(y_true == 0, y_prob[:, 0]) if len(np.unique(y_true)) == 2 else float("nan")
metrics = {
    k: round(float(v), 4)
    for k, v in {
        "val_acc": best["val_acc"],
        "test_acc": test_acc,
        "test_precision": prec,
        "test_recall": rec,
        "test_f1": f1,
        "test_roc_auc": auc,
        "test_ece": ece(y_prob, y_true),
    }.items()
}
print(json.dumps(metrics, indent=2))
print(classification_report(y_true, y_pred, labels=[0, 1], target_names=CLASSES, zero_division=0))

Left: which classes get confused. Right: the reliability diagram; points on the diagonal mean the confidence is
honest (numbers = images per bin).

In [ ]:
fig, (ax_cm, ax_rel) = plt.subplots(1, 2, figsize=(11, 4))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=[0, 1]), display_labels=CLASSES).plot(
    ax=ax_cm, cmap="Blues", colorbar=False
)
ax_cm.set_title(f"Test confusion matrix (acc {test_acc:.3f})")

conf, acc, counts = zip(*calibration_bins(y_prob, y_true), strict=True)
ax_rel.plot([0.5, 1], [0.5, 1], "--", c="gray", lw=1, label="perfectly calibrated")
ax_rel.plot(conf, acc, "o-", label="model")
for x, y, n in zip(conf, acc, counts, strict=True):
    ax_rel.annotate(str(n), (x, y), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=8)
ax_rel.set(
    title="Reliability (test, calibrated)", xlabel="confidence", ylabel="accuracy", xlim=(0.5, 1), ylim=(0, 1.08)
)
ax_rel.legend(loc="lower right")
plt.tight_layout()
plt.show()

### Mistakes and Grad-CAM

Grad-CAM weights the last convolutional feature maps by how much they push the score of a class, showing which image
regions drove a prediction. It should light up the sausage and bun, not the plate.

In [ ]:
def gradcam(x, target_cls):
    # Grad-CAM for one (1, 3, H, W) tensor -> (H, W) map in [0, 1]
    layer, store = cam_layer(model), {}
    hooks = [
        layer.register_forward_hook(lambda m, i, o: store.__setitem__("act", o)),
        layer.register_full_backward_hook(lambda m, gi, go: store.__setitem__("grad", go[0])),
    ]
    try:
        model.eval()
        model.zero_grad(set_to_none=True)
        model(x.to(device).requires_grad_(True))[0, target_cls].backward()
        weights = store["grad"].mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * store["act"]).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        return ((cam - cam.min()) / (cam.max() - cam.min() + 1e-8)).detach().float().cpu().numpy()
    finally:
        for h in hooks:
            h.remove()
        model.zero_grad(set_to_none=True)


def cam_overlay(i, cls):
    base = np.asarray(view_tf(open_image(test_x[i]))) / 255.0
    cam = gradcam(test_ds[i][0].unsqueeze(0), cls)
    return np.clip(0.55 * base + 0.45 * plt.cm.jet(cam)[..., :3], 0, 1)


wrong = np.flatnonzero(y_true != y_pred)
print(f"{len(wrong)} of {len(y_true)} test images misclassified")
if len(wrong):
    wrong = wrong[np.argsort(-y_prob[wrong].max(1))][:12]  # most confident mistakes first
    show_grid(
        [thumbnail(test_x[i]) for i in wrong],
        [f"true {CLASSES[y_true[i]]}\npred {CLASSES[y_pred[i]]} {y_prob[i].max():.2f}" for i in wrong],
        "Misclassified test images (most confident first)",
    )
    show_grid(
        [cam_overlay(i, y_pred[i]) for i in wrong],
        [f"why '{CLASSES[y_pred[i]]}'?" for i in wrong],
        "Grad-CAM: what drove each wrong prediction",
    )

hits = np.flatnonzero((y_true == 0) & (y_pred == 0))
if len(hits):
    hits = hits[np.argsort(-y_prob[hits, 0])][:6]
    show_grid(
        [cam_overlay(i, 0) for i in hits],
        [f"hotdog {y_prob[i, 0]:.2f}" for i in hits],
        "Grad-CAM on confident correct hotdogs (should light up the sausage + bun)",
    )

### Latency

Batch size 1, forward pass only, fp32: what the server reports as `latency_ms`.

In [ ]:
def benchmark(m, n_warmup, n_runs):
    m.eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device).contiguous(memory_format=MEMORY_FORMAT)
    times = []
    with torch.inference_mode():
        for _ in range(n_warmup):
            m(x)
        sync_device(device)
        for _ in range(n_runs):
            t0 = time.perf_counter()
            m(x)
            sync_device(device)
            times.append((time.perf_counter() - t0) * 1000)
    return float(np.median(times)), float(np.percentile(times, 95))


lat_p50, lat_p95 = benchmark(model, *((2, 5) if SMOKE else (10, 100)))
print(f"{ARCH} on {device}: p50 {lat_p50:.2f} ms/image, p95 {lat_p95:.2f} ms/image")

## 9. Export

Writes `hotdog_cnn.pt` (weights), `hotdog_cnn.ts.pt` (TorchScript fallback) and `model_meta.json` (architecture,
preprocessing, temperature, metrics) to `artifacts/`, where the server loads them on its next start (`./run.sh`).
The model is saved from a fresh CPU copy, so the files do not depend on the training device.

In [ ]:
print("==> Export")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

cpu_model = build_model(ARCH, num_classes=len(CLASSES)).eval()
cpu_model.load_state_dict(best["state"])
torch.save(cpu_model.state_dict(), ARTIFACTS_DIR / "hotdog_cnn.pt")

try:
    with warnings.catch_warnings():  # torch.jit is deprecated in recent torch but still works
        warnings.simplefilter("ignore")
        torch.jit.trace(cpu_model, torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)).save(str(ARTIFACTS_DIR / "hotdog_cnn.ts.pt"))
    ts_ok = True
except Exception as e:  # TorchScript is optional; the state_dict is the primary format
    print("TorchScript export failed:", e)
    ts_ok = False

meta = {
    "arch": ARCH,
    "arch_description": describe(ARCH),
    "pretrained": PRETRAINED,
    "classes": CLASSES,
    "img_size": IMG_SIZE,
    "resize_size": RESIZE_SIZE,
    "mean": IMAGENET_MEAN,
    "std": IMAGENET_STD,
    "temperature": round(TEMPERATURE, 5),
    "metrics": metrics,
    "best_epoch": best["epoch"],
    "epochs_run": len(history["val_acc"]),
    "train_minutes": round(train_minutes, 2),
    "params": count_params(cpu_model)[0],
    "latency_ms_p50": round(lat_p50, 3),
    "latency_device": device.type,
    "dataset": data_desc,
    "train_pool_by_source": dict(Counter(s.split(":")[0] for s in train_src + val_src)),
    "n_train": len(train_y),
    "n_val": len(val_y),
    "n_test": len(test_y),
    "smoke": SMOKE,
    "torch": torch.__version__,
    "torchscript": ts_ok,
    "trained_at": datetime.now(UTC).isoformat(timespec="seconds"),
}
(ARTIFACTS_DIR / "model_meta.json").write_text(json.dumps(meta, indent=2))
for f in sorted(ARTIFACTS_DIR.glob("[!.]*.*")):  # skip hidden files such as .gitkeep
    print(f"{f.name:22s} {f.stat().st_size / 1e6:7.2f} MB")

Finally, reload the files the way the server does and check that they reproduce the trained model's probabilities.

In [ ]:
reloaded = build_model(meta["arch"], num_classes=len(meta["classes"])).eval()
reloaded.load_state_dict(torch.load(ARTIFACTS_DIR / "hotdog_cnn.pt", map_location="cpu", weights_only=True))
x = torch.stack([test_ds[i][0] for i in range(min(8, len(test_ds)))])
with torch.inference_mode():
    expected = (cpu_model(x) / TEMPERATURE).softmax(1)
    assert torch.allclose((reloaded(x) / meta["temperature"]).softmax(1), expected, atol=1e-5), "state_dict mismatch"
    if ts_ok:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            scripted = torch.jit.load(str(ARTIFACTS_DIR / "hotdog_cnn.ts.pt"))
        assert torch.allclose((scripted(x) / TEMPERATURE).softmax(1), expected, atol=1e-4), "TorchScript mismatch"
print("reload check passed")